In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
import pandas as pd

In [4]:
books = pd.read_csv("books_cleaned.csv")

In [5]:
books["tagged_description"]

0       9780002005883 A NOVEL THAT READERS and critics...
1       9780002261982 A new 'Christie for Christmas' -...
2       9780006178736 A memorable, mesmerizing heroine...
3       9780006280897 Lewis' work on the nature of lov...
4       9780006280934 "In The Problem of Pain, C.S. Le...
                              ...                        
5192    9788172235222 On A Train Journey Home To North...
5193    9788173031014 This book tells the tale of a ma...
5194    9788179921623 Wisdom to Create a Life of Passi...
5195    9788185300535 This collection of the timeless ...
5196    9789027712059 Since the three volume edition o...
Name: tagged_description, Length: 5197, dtype: str

In [6]:
books["tagged_description"].to_csv("tagged_description.txt", sep="\n", index=False,header=False)

In [7]:
max_len = books["tagged_description"].str.len().max()
print(max_len)  # use this as chunk_size

5800


In [8]:
raw_documents = TextLoader("tagged_description.txt",encoding="utf-8").load()
text_splitter = CharacterTextSplitter(chunk_size=6000,chunk_overlap=0,separator="\n")
documents = text_splitter.split_documents(raw_documents)

In [9]:
documents[0]

Document(metadata={'source': 'tagged_description.txt'}, page_content='9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gi

In [10]:
print(len(documents))

468


In [11]:
import os
from langchain_community.vectorstores import Chroma
import time

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.getenv("VERTEX_AI_API_KEY")
)

BATCH_SIZE = 10
db_books = Chroma.from_documents(documents[:BATCH_SIZE], embedding=embeddings, persist_directory="book_db")

for i in range(BATCH_SIZE, len(documents), BATCH_SIZE):
    batch = documents[i:i + BATCH_SIZE]
    db_books.add_documents(batch)
    time.sleep(1)
    print(f"Processed {min(i + BATCH_SIZE, len(documents))}/{len(documents)} docs")

db_books.persist()
print("Done! chroma_db/ folder is ready.")

Processed 20/468 docs
Processed 30/468 docs
Processed 40/468 docs
Processed 50/468 docs
Processed 60/468 docs
Processed 70/468 docs
Processed 80/468 docs
Processed 90/468 docs
Processed 100/468 docs
Processed 110/468 docs
Processed 120/468 docs
Processed 130/468 docs
Processed 140/468 docs
Processed 150/468 docs
Processed 160/468 docs
Processed 170/468 docs
Processed 180/468 docs
Processed 190/468 docs
Processed 200/468 docs
Processed 210/468 docs
Processed 220/468 docs
Processed 230/468 docs
Processed 240/468 docs
Processed 250/468 docs
Processed 260/468 docs
Processed 270/468 docs
Processed 280/468 docs
Processed 290/468 docs
Processed 300/468 docs
Processed 310/468 docs
Processed 320/468 docs
Processed 330/468 docs
Processed 340/468 docs
Processed 350/468 docs
Processed 360/468 docs
Processed 370/468 docs
Processed 380/468 docs
Processed 390/468 docs
Processed 400/468 docs
Processed 410/468 docs
Processed 420/468 docs
Processed 430/468 docs
Processed 440/468 docs
Processed 450/468 d

C:\Users\ELCOT\AppData\Local\Temp\ipykernel_8216\3709474406.py:19: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db_books.persist()


In [12]:
query = "A book to teach children about Nature"

docs = db_books.similarity_search(query,k=10)

docs

[Document(metadata={'source': 'tagged_description.txt'}, page_content='9780064472777 Top Ten Reasons Samantha Madison is in Deep Trouble 10. Her big sister is the most popular girl in school 9. Her little sister is a certified genius 8. She\'s in love with her big sister\'s boyfriend 7. She got caught selling celebrity portraits in school 6. And now she\'s being forced to take art classes 5. She\'s just saved the president of the United Statesfrom an assassination attempt 4. So the whole world thinks she is a hero 3. Even though Sam knows she is far, far from being a hero 2. And now she\'s been appointed teen ambassador to the UNAnd the number-one reason Sam\'s life is over?1. The president\'s son just might be in love with her\n9780064473552 Strange things happen at Hexwood Farm. From her window, Ann Stavely watches person after person disappear through the farm\'s gate -- and never come out again. Later, in the woods nearby, she meets a tormented sorcerer, who seems to have arisen fr

In [13]:
books[books["isbn13"] == int(docs[0].page_content.split()[0].strip())]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
437,9780064472777,0064472779,All-American Girl,Meg Cabot,Juvenile Fiction,http://books.google.com/books/content?id=Fktx5...,Top Ten Reasons Samantha Madison is in Deep Tr...,2003.0,3.74,398.0,58422.0,All-American Girl,9780064472777 Top Ten Reasons Samantha Madison...


In [14]:
def get_semantic_recommendation(
        query:str,
        top_k:int = 10,
)->pd.DataFrame:
    recs = db_books.similarity_search(query,k=40)

    books_list = []

    import re
    books_list = [int(re.search(r'\d+', rec.page_content.split()[0]).group()) for rec in recs]

    return books[books["isbn13"].isin(books_list)].head(top_k)



In [15]:
get_semantic_recommendation("A book to teach children about Nature")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
76,9780007204465,0007204469,Tropic of Cancer,Henry Miller,Fiction,http://books.google.com/books/content?id=ProgR...,"Miller's groundbreaking first novel, banned in...",2005.0,3.69,336.0,709.0,Tropic of Cancer,9780007204465 Miller's groundbreaking first no...
97,9780060090371,0060090375,Lucy Sullivan Is Getting Married,Marian Keyes,Fiction,http://books.google.com/books/content?id=qLl-3...,What happens when a psychic tells Lucy that sh...,2002.0,3.76,640.0,42096.0,Lucy Sullivan Is Getting Married,9780060090371 What happens when a psychic tell...
105,9780060256579,0060256575,The Missing Piece Meets the Big O,Shel Silverstein,Juvenile Fiction,http://books.google.com/books/content?id=-m4gw...,The missing piece sat alone waiting for someon...,1981.0,4.33,104.0,10010.0,The Missing Piece Meets the Big O,9780060256579 The missing piece sat alone wait...
315,9780060936662,0060936665,Smart Discipline(R),Larry Koenig,Family & Relationships,http://books.google.com/books/content?id=Bpo2r...,"Larry J. Koenig, Ph.D., creator of the hugely ...",2004.0,3.99,208.0,12.0,"Smart Discipline(R): Fast, Lasting Solutions f...","9780060936662 Larry J. Koenig, Ph.D., creator ..."
385,9780061150142,0061150142,The Pact,Jodi Picoult,Fiction,http://books.google.com/books/content?id=VDCvq...,Until the phone calls came at three o'clock on...,2006.0,4.01,512.0,237587.0,The Pact: A Love Story,9780061150142 Until the phone calls came at th...
395,9780062508867,0062508865,Muhammad,Karen Armstrong,Biography & Autobiography,http://books.google.com/books/content?id=FbDVD...,This vivid and detailed biography strips away ...,1993.0,4.15,304.0,4795.0,Muhammad,9780062508867 This vivid and detailed biograph...
407,9780064404419,0064404412,The Rainbow People,Laurence Yep,Juvenile Fiction,http://books.google.com/books/content?id=5AHwq...,"""Culled from 69 stories collected in a [1930s]...",1992.0,3.75,208.0,202.0,The Rainbow People,"9780064404419 ""Culled from 69 stories collecte..."
420,9780064408677,0064408671,The Trumpet of the Swan,E. B. White,Juvenile Fiction,http://books.google.com/books/content?id=2lybT...,"Swan Song Like the rest of his family, Louis i...",2000.0,4.07,252.0,61304.0,The Trumpet of the Swan,9780064408677 Swan Song Like the rest of his f...
430,9780064435260,0064435261,A Little Prairie House,Laura Ingalls Wilder,Juvenile Fiction,http://books.google.com/books/content?id=pRSju...,"Long, long ago, a little girl named Laura Inga...",1999.0,4.19,32.0,1533.0,A Little Prairie House,"9780064435260 Long, long ago, a little girl na..."
437,9780064472777,0064472779,All-American Girl,Meg Cabot,Juvenile Fiction,http://books.google.com/books/content?id=Fktx5...,Top Ten Reasons Samantha Madison is in Deep Tr...,2003.0,3.74,398.0,58422.0,All-American Girl,9780064472777 Top Ten Reasons Samantha Madison...
